# Portfolio Project: Online Retail Exploratory Data Analysis with Python

## Overview

In this project, you will step into the shoes of an entry-level data analyst at an online retail company, helping interpret real-world data to help make a key business decision.

## Case Study
In this project, you will be working with transactional data from an online retail store. The dataset contains information about customer purchases, including product details, quantities, prices, and timestamps. Your task is to explore and analyze this dataset to gain insights into the store's sales trends, customer behavior, and popular products. 

By conducting exploratory data analysis, you will identify patterns, outliers, and correlations in the data, allowing you to make data-driven decisions and recommendations to optimize the store's operations and improve customer satisfaction. Through visualizations and statistical analysis, you will uncover key trends, such as the busiest sales months, best-selling products, and the store's most valuable customers. Ultimately, this project aims to provide actionable insights that can drive strategic business decisions and enhance the store's overall performance in the competitive online retail market.

## Project Objectives
1. Describe data to answer key questions to uncover insights
2. Perform data visualization to gain insights into the dataset. Generate appropriate plots, such as histograms, scatter plots, or bar plots, to visualize different aspects of the data.
3. Gain valuable insights that will help improve online retail performance
4. Provide analytic insights and data-driven recommendations

## Dataset

The dataset you will be working with is the "Online Retail" dataset. It contains transactional data of an online retail store from 2010 to 2011. The dataset is available as a .xlsx file named `Online Retail.xlsx`. This data file is already included in the Coursera Jupyter Notebook environment, however if you are working off-platform it can also be downloaded [here](https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx).

The dataset contains the following columns:

- InvoiceNo: Invoice number of the transaction
- StockCode: Unique code of the product
- Description: Description of the product
- Quantity: Quantity of the product in the transaction
- InvoiceDate: Date and time of the transaction
- UnitPrice: Unit price of the product
- CustomerID: Unique identifier of the customer
- Country: Country where the transaction occurred

## Tasks

You may explore this dataset in any way you would like - however if you'd like some help getting started, here are a few ideas:

1. Perform data visualization to gain insights into the dataset. Generate appropriate plots, such as histograms, scatter plots, or bar plots, to visualize different aspects of the data.
2. Analyze the sales trends over time. Identify the busiest months and days of the week in terms of sales.
4. Draw conclusions and summarize your findings from the exploratory data analysis.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path = Path.cwd()) -> Path:
    for path in [start, *start.parents]:
        if (path / "README.md").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Project root not found")

BASE_DIR = find_project_root()
print(f'Working base set at {BASE_DIR}')

if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

DATA_DIR = BASE_DIR / "data"
retail_path = DATA_DIR / 'Online Retail Cleaned.xlsx'

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.visualization import plot_raw_log_boxplot

## Task 1: Perform data visualization to gain insights into the dataset

In [ ]:
df = pd.read_excel(retail_path)
display(df.head(5))

### Create analysis variables

The cleaned dataset is still at line-item level: each row represents one product line inside an invoice. For multivariate analysis, I first create invoice-level variables so revenue can be analyzed at the level of a complete purchase rather than individual product rows.


In [ ]:
df['sales'] = df['Quantity'] * df['UnitPrice']
df['is_return'] = df['Quantity'] < 0

invoice_sizes = df.groupby('InvoiceNo').size()
large_inv_ids = invoice_sizes[invoice_sizes > 100].index
df['is_large_invoice'] = df['InvoiceNo'].isin(large_inv_ids)

In [ ]:
invoice_df = (
    df.groupby('InvoiceNo')
    .agg(
        invoice_date=("InvoiceDate", "first"),
        customer_id=("CustomerID", "first"),
        country=("Country", "first"),
        line_count=("StockCode", "size"),
        unique_products=("StockCode", "nunique"),
        total_quantity=('Quantity', 'sum'),
        revenue=('sales', 'sum'),
        avg_unit_price=('UnitPrice', 'mean'),
        avg_quantity=('Quantity', 'mean'),
        has_negative_q=('Quantity', lambda x: (x < 0).any()),
        is_uk=('Country', lambda x: (x == 'united kingdom').any())
    ).reset_index()
)

invoice_df["weighted_avg_unit_price"] = (
    invoice_df["revenue"] / invoice_df["total_quantity"]
)

invoice_df["revenue_per_line"] = (
    invoice_df["revenue"] / invoice_df["line_count"]
)

invoice_df['month'] = invoice_df['invoice_date'].dt.month
invoice_df['day_of_week'] = invoice_df['invoice_date'].dt.dayofweek


purchase_invoices = invoice_df[
    (invoice_df["revenue"] > 0) &
    (~invoice_df["has_negative_q"])
]

purchase_invoices.to_excel(DATA_DIR / 'Online Retail purchases.xlsx', index=False)

For invoice-level purchase analysis, invoices containing negative quantities were excluded. This keeps the main revenue and basket-size metrics focused on standard purchases instead of mixing purchases with returns, cancellations, or adjustments.

This is an analytical choice, not a claim that negative quantities are invalid data. Returns and cancellations should be analyzed separately if the business question is about refunds or operational issues.


### Correlation Matrix

The first correlation matrix summarizes relationships between invoice-level purchase variables. The main focus is revenue, because it is the most direct business outcome in this notebook.


In [ ]:
corr_cols = [
    'line_count', 'total_quantity', 'avg_unit_price',
    'avg_quantity', 'revenue', 'weighted_avg_unit_price', 'revenue_per_line'
]

corr_matrix = purchase_invoices[corr_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1, square=True)
plt.show()

**Interpretation**

Invoice revenue is mainly associated with purchase volume. `total_quantity` has the strongest correlation with `revenue`, while `line_count` and `unique_products` have much weaker relationships. This suggests that high-revenue invoices are not necessarily more diverse baskets; they are more likely invoices with larger quantities.

`avg_unit_price` and `weighted_avg_unit_price` have very weak correlation with revenue, which suggests that high-revenue invoices are not primarily explained by expensive products. Instead, revenue appears to be driven more by quantity than by price level.

A limitation is that several variables are mathematically related. For example, `avg_quantity`, `revenue_per_line`, and `weighted_avg_unit_price` are derived from quantity, revenue, and line count. The matrix is useful for describing invoice structure, but it should not be interpreted as causal evidence.


In [ ]:
uk_purchases = purchase_invoices[purchase_invoices["country"] == "united kingdom"]
non_uk_purchases = purchase_invoices[purchase_invoices["country"] != "united kingdom"]

corr_matrix = uk_purchases[corr_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1, square=True)
plt.show()

In [ ]:
corr_matrix = non_uk_purchases[corr_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1, square=True)
plt.show()

**Segment comparison: UK vs non-UK**

The UK and non-UK segments show different invoice patterns. In both groups, revenue is strongly related to `total_quantity`, so purchase volume remains the main revenue driver.

In the UK segment, revenue is more strongly associated with `avg_quantity` and `revenue_per_line`. This suggests that high-revenue UK invoices often come from concentrated baskets with many units per line.

Outside the UK, `line_count` and `unique_products` have a stronger relationship with revenue than they do in the UK. This suggests that non-UK high-revenue invoices may depend more on broader baskets, while UK high-revenue invoices are more concentrated in quantity per line.

`line_count` and `unique_products` are identical in this dataset, so `line_count` can be used as a practical proxy for basket diversity at invoice level.


### Scatterplots

The scatterplots are used to visually check the relationships suggested by the correlation matrix. Because revenue and quantity variables are highly right-skewed, logarithmic scales are used where appropriate to make the dense region of normal purchases easier to inspect without removing large valid invoices.


#### Revenue vs Average Quantity


In [ ]:
sns.scatterplot(data=purchase_invoices, x="revenue", y="avg_quantity", hue='is_uk', alpha=0.35)
plt.xscale("log")
plt.yscale("log")
plt.tight_layout()
plt.show()

In [ ]:
avg_q_no_outliers = purchase_invoices[purchase_invoices['avg_quantity'] < 10000]
sns.scatterplot(data=avg_q_no_outliers, x="revenue", y="avg_quantity", hue='is_uk')
plt.tight_layout()
plt.show()

**Interpretation**

High average quantity is not sufficient by itself to produce high revenue. Some invoices have high average quantity but relatively low revenue, which can happen when products are inexpensive. Revenue increases most clearly when quantity combines with enough product lines and/or enough unit value.

The pattern supports the idea that high-revenue invoices often depend on volume, but volume must still be interpreted together with price and basket structure.


#### Revenue vs Average Unit Price


In [ ]:
sns.scatterplot(data=purchase_invoices, x="revenue", y="avg_unit_price", hue='is_uk', alpha=0.35)
plt.xscale("log")
plt.yscale("log")
plt.tight_layout()
plt.show()

In [ ]:
avg_u_price_no_outliers = purchase_invoices[purchase_invoices['avg_unit_price'] < 7000]
sns.scatterplot(data=avg_u_price_no_outliers, x="revenue", y="avg_unit_price", hue='is_uk')
plt.tight_layout()
plt.show()

**Interpretation**

High unit prices are neither common nor necessary for high invoice revenue. Most high-revenue invoices appear in low-to-moderate average price ranges, suggesting that revenue is mainly volume-driven.

This does not mean price is irrelevant; it means that, at invoice level, higher average price is not the main pattern associated with higher revenue in this dataset.


#### Revenue vs Total Quantity


In [ ]:
sns.scatterplot(data=purchase_invoices, x="revenue", y="total_quantity", hue='is_uk', alpha=0.35)
plt.xscale("log")
plt.yscale("log")
plt.tight_layout()
plt.show()

In [ ]:
total_q_no_outliers = purchase_invoices[purchase_invoices['total_quantity'] < 50000]
sns.scatterplot(data=total_q_no_outliers, x="revenue", y="total_quantity", hue='is_uk')
plt.tight_layout()
plt.show()

**Interpretation**

Total quantity has the clearest positive relationship with revenue. This is expected because invoice revenue is directly built from quantity and unit price, but the plot confirms that purchase volume is the most visible driver of revenue variation.

This relationship should still be interpreted descriptively: it does not explain why customers buy larger quantities, only that larger quantities are strongly associated with higher invoice revenue.


#### Revenue vs Basket Diversity


In [ ]:
sns.scatterplot(data=purchase_invoices, x="revenue", y="line_count", hue='is_uk', alpha=0.35)
plt.xscale("log")
plt.yscale("log")
plt.tight_layout()
plt.show()

In [ ]:
sns.scatterplot(data=purchase_invoices, x="revenue", y="unique_products", hue='is_uk')
plt.tight_layout()
plt.show()

`unique_products` and `line_count` show the same pattern. This means repeated product codes within the same invoice are not common enough to make these variables meaningfully different for this analysis.


**Interpretation**

Basket diversity has a weaker relationship with revenue than total quantity. Invoices with many lines do not necessarily generate high revenue, especially in the UK segment.

For non-UK invoices, the relationship appears somewhat stronger, suggesting that international high-revenue invoices may depend more on broader baskets than UK invoices. This is a useful segment difference to revisit in the final insights notebook.


#### Revenue vs Revenue per Line


In [ ]:
sns.scatterplot(data=purchase_invoices, x="revenue", y="revenue_per_line", hue='is_uk', alpha=0.35)
plt.xscale("log")
plt.yscale("log")
plt.tight_layout()
plt.show()

#### Revenue vs Month


In [ ]:
month_names = {
    1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr",
    5: "May", 6: "Jun", 7: "Jul", 8: "Aug",
    9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec",
}
month_order = list(month_names.values())
purchase_invoices['month_name'] = (
    purchase_invoices['month']
    .map(month_names)
    .astype(pd.CategoricalDtype(categories=month_order, ordered=True))
)

sns.scatterplot(data=purchase_invoices, x="revenue", y="month", hue='is_uk')
plt.tight_layout()
plt.show()

#### Revenue vs Day of Week


In [ ]:
day_names = {
    0: "Monday",
    1: "Tuesday",
    2: "Wednesday",
    3: "Thursday",
    4: "Friday",
    5: "Saturday",
    6: "Sunday",
}

day_order = list(day_names.values())
purchase_invoices['day_name'] = (
    purchase_invoices['day_of_week']
    .map(day_names)
    .astype(pd.CategoricalDtype(categories=day_order, ordered=True))
)

sns.scatterplot(data=purchase_invoices, x="revenue", y="day_name", hue='is_uk')
plt.tight_layout()
plt.show()

**Interpretation**

Revenue activity is concentrated on weekdays, with Sunday showing lower activity and Saturday apparently absent from the cleaned purchase dataset. Differences between weekdays appear more related to transaction volume and high-revenue invoices than to a clear change in typical invoice value.

Because day of week is a discrete time variable, aggregated tables or boxplots are more informative than scatterplots for final conclusions.


#### Revenue vs Weighted Average Unit Price


In [ ]:
sns.scatterplot(data=purchase_invoices, x="revenue", y="weighted_avg_unit_price", hue='is_uk', alpha=0.35)
plt.xscale("log")
plt.yscale("log")
plt.tight_layout()
plt.show()

### Revenue Distribution

The next plots compare invoice revenue across calendar groups. The goal is to understand whether some months or weekdays have different revenue distributions, not only different total revenue.


#### Monthly Revenue Distribution


In [ ]:
plot_raw_log_boxplot(df=purchase_invoices, x='month', y='revenue', title='Revenue by month')

In [ ]:
month_outliers = purchase_invoices[purchase_invoices['revenue'] < 30000]

plot_raw_log_boxplot(df=month_outliers, x='month', y='revenue', title='Revenue by month')

**Interpretation**

Revenue is highly right-skewed in every month. A small number of high-revenue invoices make raw-scale boxplots difficult to read, so a log scale is useful to compare the typical invoice range without removing large valid purchases.

The typical invoice value does not appear to change dramatically month to month. Stronger months are likely driven by a combination of more invoices and occasional high-revenue purchases rather than a completely different typical invoice profile.


#### Revenue Distribution by Day of Week


In [ ]:
plot_raw_log_boxplot(
    df=purchase_invoices, x='day_name',
    y='revenue', title='Revenue by day of week'
)

In [ ]:
dow_outliers = purchase_invoices[purchase_invoices['revenue'] < 30000]

plot_raw_log_boxplot(
    df=dow_outliers, x='day_of_week',
    y='revenue', title='Revenue by day of week'
)

# Conclusions

## Correlations

The correlation analysis suggests that invoice revenue is mainly driven by purchase volume rather than unit price. `total_quantity` has the strongest relationship with revenue across all segments, while `avg_unit_price` and `weighted_avg_unit_price` show very weak correlation with revenue.

In the UK segment, revenue is more strongly associated with `avg_quantity` and `revenue_per_line`, suggesting that high-revenue invoices often come from large quantities within relatively concentrated baskets. Outside the UK, revenue is even more strongly correlated with `total_quantity`, but `line_count` has a higher correlation than in the UK. This suggests that non-UK high-revenue invoices may rely more on broader baskets, while UK high-revenue invoices are more concentrated in quantity per line.

These results should be interpreted as descriptive patterns, not causal effects, because several variables are mathematically derived from quantity, line count, and revenue.

## Scatterplots

The scatterplots support the correlation findings. Revenue is most clearly associated with total quantity, while average unit price shows little relationship with revenue. High-revenue invoices are usually not explained by expensive products, but by volume.

Basket diversity, measured by `line_count` or `unique_products`, has a weaker relationship with revenue overall, although it appears somewhat more relevant outside the UK.

## Calendar Patterns

Monthly and weekday revenue distributions are highly right-skewed. Large invoices strongly affect the visual scale, so log-scale plots are more useful for comparing typical invoice behavior. The typical invoice revenue does not appear to vary as strongly as total activity, meaning stronger periods may be driven more by transaction volume and upper-tail purchases than by a large shift in the median invoice.

## Limitations

This notebook focuses on purchase invoices and excludes invoices with negative quantities. Returns, cancellations, and adjustments may have different patterns and should be analyzed separately. The analysis is descriptive and does not establish causality. Several invoice-level variables are derived from each other, so correlations should be interpreted as structure summaries rather than independent drivers.

## Future Work

- Compare Pearson and Spearman correlations to check whether the relationships are stable under strong skew and outliers.
- Repeat the analysis excluding the top 1% of revenue to check whether the main patterns are dominated by a small number of very large invoices.
- Use the final insights notebook to answer business questions directly using selected metrics from this notebook.
